In [1]:
# Paper baseline (Notebook 05 protocol) on Part D.
# SMOKE=1 (default) = 1 run, 1 fold, XGB only. sbatch sets SMOKE=0.
# LABEL_COL picks the label: fraud (paper), fraud_h2, fraud_h3.
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

SMOKE     = os.environ.get("SMOKE", "1") == "1"
LABEL_COL = os.environ.get("LABEL_COL", "fraud")
N_JOBS    = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))   # never -1 on Amarel

DATA_DIR = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
LABELED  = DATA_DIR / "processed/part_d/labeled/part_d_A_labeled_2019_2023.parquet"
RES_DIR  = DATA_DIR / "results/part_d"; RES_DIR.mkdir(parents=True, exist_ok=True)
TAG      = f"nb08_{LABEL_COL}" + ("_smoke" if SMOKE else "")
RES_CSV  = RES_DIR / f"{TAG}_fold_results.csv"
print("SMOKE:", SMOKE, "| label:", LABEL_COL, "| N_JOBS:", N_JOBS, "| results →", RES_CSV.name)

SMOKE: False | label: fraud_h2 | N_JOBS: 16 | results → nb08_fraud_h2_fold_results.csv


## Settings

Same as Notebook 05: 6 runs × 5 folds, k-fold-by-NPI stratified by ever-fraud, XGB depth 4, RF depth 16, threshold = training-fold prior.

Datasets: only **A** (aggregated) exists for Part D right now. When the enriched file (with Rahi's by-Provider features) is labeled, point `LABELED` at it and SbP / AE are picked up automatically.

In [2]:
N_RUNS, N_FOLDS = (1, 5) if SMOKE else (6, 5)
FOLDS_TO_RUN    = [0] if SMOKE else list(range(N_FOLDS))
LEARNERS        = ["XGB"] if SMOKE else ["XGB", "RF"]

def make_model(name, seed):
    if name == "XGB":
        return XGBClassifier(max_depth=4, n_jobs=N_JOBS, random_state=seed,
                             tree_method="hist", eval_metric="logloss")
    return RandomForestClassifier(max_depth=16, n_jobs=N_JOBS, random_state=seed)

In [3]:
df = pd.read_parquet(LABELED)
LABEL_COLS = [c for c in df.columns if c == "fraud" or c.startswith("fraud_h")]
assert LABEL_COL in LABEL_COLS, LABEL_COLS
print(df.shape, "| label rows:", int(df[LABEL_COL].sum()))

KEYS  = ["npi", "year"] + LABEL_COLS
CATS  = ["provider_type"]
BASES = ["total_claims", "total_30day_fills", "total_day_supply", "total_drug_cost", "total_benes"]
STATS = ["min", "max", "median", "mean", "sum", "std"]
AGG   = [f"{b}_{s}" for b in BASES for s in STATS]
SBP   = [c for c in df.columns if c not in KEYS + CATS + AGG]
assert all(c in df.columns for c in AGG)
print("AGG:", len(AGG), "| SbP:", len(SBP))

DATASETS = ["SbP", "A", "AE"] if SBP else ["A"]

(5148303, 36) | label rows: 958
AGG: 30 | SbP: 0


In [4]:
cat_df = pd.get_dummies(df["provider_type"], prefix="ptype", dtype=np.float32)
COMMON = list(cat_df.columns)
FEATS  = AGG + SBP + COMMON
X_all  = np.hstack([df[AGG + SBP].to_numpy(np.float32), cat_df.to_numpy(np.float32)])
y_all  = df[LABEL_COL].to_numpy(np.int8)
del cat_df

col_idx = {f: i for i, f in enumerate(FEATS)}
DS_COLS = {"SbP": [col_idx[c] for c in SBP + COMMON],
           "A":   [col_idx[c] for c in AGG + COMMON],
           "AE":  [col_idx[c] for c in AGG + SBP + COMMON]}
for k in DATASETS: print(k, "features:", len(DS_COLS[k]))
print("X_all:", X_all.shape, f"{X_all.nbytes/1e9:.1f} GB")

A features: 240
X_all: (5148303, 240) 4.9 GB


In [5]:
npi = df["npi"].to_numpy()
npi_ever_fraud = df.groupby("npi")[LABEL_COL].max()

def assign_folds(seed):
    rng = np.random.default_rng(seed)
    parts = []
    for cls in (0, 1):
        ids = rng.permutation(npi_ever_fraud.index[npi_ever_fraud == cls].to_numpy())
        parts.append(pd.Series(np.arange(len(ids)) % N_FOLDS, index=ids))
    return pd.Series(npi).map(pd.concat(parts)).to_numpy()

f0 = assign_folds(0)
chk = pd.DataFrame({"npi": npi, "fold": f0, "y": y_all})
assert chk.groupby("npi").fold.nunique().max() == 1, "NPI leaks across folds"
print(chk.groupby("fold").agg(rows=("y", "size"), fraud_rows=("y", "sum"),
      fraud_npis=("npi", lambda s: s[chk.loc[s.index, "y"] == 1].nunique())))
del chk

         rows  fraud_rows  fraud_npis
fold                                 
0     1029968         197         121
1     1029079         189         120
2     1029321         188         120
3     1029685         183         120
4     1030250         201         120


In [6]:
def evaluate(y_true, p, thr):
    pred = (p >= thr).astype(np.int8)
    tp = ((pred == 1) & (y_true == 1)).sum(); fn = ((pred == 0) & (y_true == 1)).sum()
    tn = ((pred == 0) & (y_true == 0)).sum(); fp = ((pred == 1) & (y_true == 0)).sum()
    tpr, tnr = tp / (tp + fn), tn / (tn + fp)
    return {"AUC": roc_auc_score(y_true, p), "AUPRC": average_precision_score(y_true, p),
            "TPR": tpr, "TNR": tnr, "GMean": np.sqrt(tpr * tnr), "threshold": thr}

In [7]:
done = set()
if RES_CSV.exists():
    prev = pd.read_csv(RES_CSV)
    done = set(zip(prev.run, prev.fold, prev.dataset, prev.learner))
print("Already done:", len(done))

for run in range(N_RUNS):
    folds = assign_folds(run)
    for k in FOLDS_TO_RUN:
        te = np.where(folds == k)[0]; tr = np.where(folds != k)[0]
        y_tr, y_te = y_all[tr], y_all[te]
        thr = y_tr.mean()                          # prior probability threshold
        for ds in DATASETS:
            cols = DS_COLS[ds]
            X_tr = X_all[np.ix_(tr, cols)]; X_te = X_all[np.ix_(te, cols)]
            for lr in LEARNERS:
                if (run, k, ds, lr) in done: continue
                t0 = time.time()
                m = make_model(lr, seed=run).fit(X_tr, y_tr)
                p = m.predict_proba(X_te)[:, 1]
                row = {"run": run, "fold": k, "dataset": ds, "learner": lr, "label": LABEL_COL,
                       **evaluate(y_te, p, thr), "fit_sec": round(time.time() - t0, 1),
                       "test_pos": int(y_te.sum())}
                pd.DataFrame([row]).to_csv(RES_CSV, mode="a", header=not RES_CSV.exists(), index=False)
                print(f"run{run} fold{k} {ds:>3} {lr}: AUC={row['AUC']:.4f} "
                      f"AUPRC={row['AUPRC']:.4f} G={row['GMean']:.4f} ({row['fit_sec']}s)")
            del X_tr, X_te

Already done: 0


run0 fold0   A XGB: AUC=0.7166 AUPRC=0.0005 G=0.6442 (23.2s)


run0 fold0   A RF: AUC=0.7087 AUPRC=0.0005 G=0.6406 (142.6s)


run0 fold1   A XGB: AUC=0.7278 AUPRC=0.0009 G=0.6669 (23.0s)


run0 fold1   A RF: AUC=0.7175 AUPRC=0.0035 G=0.6459 (141.0s)


...

run5 fold2   A RF: AUC=0.7421 AUPRC=0.0007 G=0.6914 (142.1s)


run5 fold3   A XGB: AUC=0.7386 AUPRC=0.0006 G=0.6710 (24.3s)


run5 fold3   A RF: AUC=0.6949 AUPRC=0.0005 G=0.6382 (143.4s)


run5 fold4   A XGB: AUC=0.7587 AUPRC=0.0010 G=0.6995 (25.2s)


run5 fold4   A RF: AUC=0.7022 AUPRC=0.0005 G=0.6349 (141.8s)


In [8]:
res = pd.read_csv(RES_CSV)
print("Results per combo:", res.groupby(["dataset", "learner"]).size().to_dict())

def ci(s):
    m = s.mean(); h = 1.96 * s.std(ddof=1) / np.sqrt(len(s)) if len(s) > 1 else np.nan
    return f"{m:.4f} ({m-h:.4f}, {m+h:.4f})"

for metric in ["AUC", "AUPRC", "GMean"]:
    t = res.pivot_table(index="dataset", columns="learner", values=metric, aggfunc=ci).reindex(DATASETS)
    print(f"\n{metric} mean (95% CI), label={LABEL_COL}\n", t)
    t.to_csv(RES_DIR / f"{TAG}_{metric}.csv")

Results per combo: {('A', 'RF'): 30, ('A', 'XGB'): 30}

AUC mean (95% CI), label=fraud_h2
 learner                       RF                      XGB
dataset                                                  
A        0.7219 (0.7151, 0.7286)  0.7434 (0.7374, 0.7493)

AUPRC mean (95% CI), label=fraud_h2
 learner                       RF                      XGB
dataset                                                  
A        0.0008 (0.0005, 0.0010)  0.0009 (0.0007, 0.0010)

GMean mean (95% CI), label=fraud_h2
 learner                       RF                      XGB
dataset                                                  
A        0.6621 (0.6541, 0.6701)  0.6751 (0.6671, 0.6832)


In [9]:
# Diagnostic (paper Fig. 2): row-level k-fold vs k-fold-by-NPI, fold 0, XGB.
# Train AUC tells weak signal (low train AUC) from poor generalization (high train, low test).
cols = DS_COLS["AE" if "AE" in DATASETS else "A"]
row_folds = np.random.default_rng(0).permutation(len(y_all)) % N_FOLDS
for name, folds in [("row-level", row_folds), ("by-NPI", assign_folds(0))]:
    tr = np.where(folds != 0)[0]; te = np.where(folds == 0)[0]
    m = make_model("XGB", 0).fit(X_all[np.ix_(tr, cols)], y_all[tr])
    auc_te = roc_auc_score(y_all[te], m.predict_proba(X_all[np.ix_(te, cols)])[:, 1])
    auc_tr = roc_auc_score(y_all[tr], m.predict_proba(X_all[np.ix_(tr, cols)])[:, 1])
    print(f"{name:>9}: train AUC={auc_tr:.4f} | test AUC={auc_te:.4f}")

row-level: train AUC=0.9457 | test AUC=0.7629


   by-NPI: train AUC=0.9577 | test AUC=0.7166


In [10]:
# AUC by year on fold 0 (XGB, A): does performance fall toward 2023?
tr = np.where(f0 != 0)[0]; te = np.where(f0 == 0)[0]
m = make_model("XGB", 0).fit(X_all[np.ix_(tr, DS_COLS["A"])], y_all[tr])
p = m.predict_proba(X_all[np.ix_(te, DS_COLS["A"])])[:, 1]
yr = df["year"].to_numpy()[te]
print(pd.DataFrame([{"year": y, "fraud_rows": int(y_all[te][yr == y].sum()),
                     "AUC": roc_auc_score(y_all[te][yr == y], p[yr == y])}
                    for y in sorted(set(yr)) if 0 < y_all[te][yr == y].sum() < (yr == y).sum()]).round(4))

   year  fraud_rows     AUC
0  2019          39  0.7229
1  2020          37  0.7076
2  2021          36  0.6755
3  2022          44  0.7268
4  2023          41  0.7417
